# Hubs for Free: one-click replication

Runs on Colab or Kaggle (CPU is enough for every step except the optional model runs). Each cell prints the numbers that the repository's committed artifacts report; the comparison targets are listed at the end.

Repository: https://github.com/StefanOjanen/hubs-for-free

In [ ]:
!git clone -q https://github.com/StefanOjanen/hubs-for-free.git && cd hubs-for-free && git log --oneline -1

In [ ]:
%cd hubs-for-free
!pip install -q -r requirements-ci.txt && pip install -q -e . && pip install -q torch transformers datasets 2>/dev/null | tail -1

## 1. The coordinator that random matrices produce (one second)

In [ ]:
!hubsfree demo

## 2. Surrogate invariants and toolkit tests

In [ ]:
!python -m pytest -q tests

## 3. The synthetic battery regenerates (under a minute on CPU) and matches the committed results.json

In [ ]:
!python experiments.py > /dev/null && python check_results.py

## 4. Anchor check on the development model (downloads Qwen2.5-0.5B, about 1 GB)

Expected (CPU or GPU, float32): layer 2 `r1_real 0.781`, layer 11 `r1_real -0.2001`, layer 20 `r1_real -0.0633`, `zmed_real 10.9165` or `10.9166`; see `alignment_study/RUNLOG.md`, local platform entry.

In [ ]:
!HUBSFREE_DEVICE=auto python alignment_study/scale_round_v2.py --validate 2>&1 | grep VALIDATE

## 5. Optional: audit a layer of any Hugging Face model with the battery

Extracts one layer's attention maps and reports the percentile of each built-in statistic in each null family, with the fraction of the effect each null reproduces.

In [ ]:
!hubsfree extract --model Qwen/Qwen2.5-0.5B --layer 11 --out maps.npy && hubsfree audit maps.npy --draws 100

## 6. Optional, GPU: one training-dynamics cell (Pythia-160m, step 143000; about 300 MB)

Re-computes one cell of `alignment_study/dynamics/pythia-160m_step143000.json`; the printed per-layer values should match that file to the fourth decimal in sink mass, shared energy and cosine.

In [ ]:
!python - <<'EOF'
import json, sys
sys.path.insert(0, 'alignment_study'); sys.path.insert(0, '.')
import dynamics_round as D
cell = D.run_cell('EleutherAI/pythia-160m', 143000)
ref = json.load(open('alignment_study/dynamics/pythia-160m_step143000.json'))['rows']
for r, q in zip(cell['rows'], ref):
    print(f"L{r['layer']:2d} smass {r['smass']:.4f} (ref {q['smass']:.4f})  sharedE {r['sharedE']:.4f} (ref {q['sharedE']:.4f})  cosS {r['cosS']:.4f} (ref {q['cosS']:.4f})")
EOF

## Comparison targets

- `check_results.py` prints `results.json regenerates within tolerance` when every number in `results.json` regenerates.
- The anchor values above are the ones every round of this repository was validated against.
- Differences beyond the fourth decimal on a GPU in float32 would be a finding worth reporting as an issue.